## Instruction Finetuning 
<div class="alert alert-info" role="alert">
Instruction fine-tuning is a process of training a pre-trained AI model, especially a large language model (LLM), on a dataset of instructions and their corresponding desired outputs. This helps the model better understand and follow human instructions to perform specific tasks, improving its ability to generalize and respond to novel prompts across a wide range of domains. 
</div>

In [5]:
import json
import os
import urllib
import ssl

def download_file(url, file_path):
    ssl_context = ssl.create_default_context() # Create an SSL context that does not verify certificates
    ssl_context.check_hostname = False # Disable hostname checking
    ssl_context.verify_mode = ssl.CERT_NONE # Disable certificate verification

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url, context = ssl_context) as response:
            text_data = response.read().decode('utf-8')
        with open(file_path, 'w', encoding = 'utf-8') as f:
            f.write(text_data)
    else:
        with open(file_path, 'r', encoding= 'utf-8') as f:
            text_data = f.read()
    with open(file_path, 'r', encoding= 'utf-8') as f:
        data = json.load(f)
    return data

file_path = 'alpaca_data.json'
data_url = 'https://raw.githubusercontent.com/tatsu-lab/stanford_alpaca/main/alpaca_data.json'
data = download_file(data_url, file_path)
data[:5]  # Display the first two entries




[{'instruction': 'Give three tips for staying healthy.',
  'input': '',
  'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'},
 {'instruction': 'What are the three primary colors?',
  'input': '',
  'output': 'The three primary colors are red, blue, and yellow.'},
 {'instruction': 'Describe the structure of an atom.',
  'input': '',
  'output': 'An atom is made up of a nucleus, which contains protons and neutrons, surrounded by electrons that travel in orbits around the nucleus. The protons and neutrons have a positive charge, while the electrons have a negative charge, resulting in an overall neutral atom. The number of each particle determines the atomic number and the type of atom.'},
 {'instruction': 'How can we reduce air pollution?',
  'input': '',
  'output': 'There are a number of ways to reduce air pollution, such

In [ ]:
len(data)

52002

In [ ]:
data[50]

{'instruction': 'Edit the following sentence to make it more concise.',
 'input': 'He ran to the bus stop in order to catch the bus that was due to arrive in five minutes.',
 'output': 'He ran to the bus stop, due to arrive in five minutes.'}

We are using the alpaca dataset for instruction finetuning. The dataset contains 52,000 instruction-following examples generated from OpenAI's text-davinci-003 model.

#### Converting the dataset into alpaca format

In [8]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n {entry["instruction"]}"
    )

    input_text = f"\n\n### Input:\n {entry["input"]}" if entry["input"] else ""

    return instruction_text + input_text

model_input = format_input(data[112])
desired_output = f"\n\n### Response:\n {data[10]["output"]}"

print(model_input + desired_output)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
 Identify the incorrect word and suggest a better version.

### Input:
 The waitress served a humonguous burger.

### Response:
 Julius Caesar was assassinated by a group of up to 60 conspirators, led by Gaius Cassius Longinus and Marcus Junius Brutus, in the Senate House on the Ides of March (15 March) of 44 BC.


In [10]:
## partioning the dataset into train and test sets
train_portion = int(len(data) * 0.85)
test_portion = int(len(data) * 0.1)
val_portion = len(data) - train_portion - test_portion

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

## Converting the data into batch format for training

1. Format the data using prompt template
2. Tokenize the formated data 
3. Adjust the same lenght with padding tokens 
4. Create target token IDs for training 
5. replace padding tokens with placeholder -100(used in pytorch to ignore certain tokens in loss computation

)

In [ ]:
## we will be using pytorch Dataset to curate the data for training

from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        self.encoded_text = []

        for entry in data:
            instruction_and_input = format_input(entry)
            response_text = f"\n\n### Response:\n{entry['output']}"
            full_text = instruction_and_input + response_text

            self.encoded_text.append(
                tokenizer.encode(full_text)
            )
    
    def __getitem__(self, index):
        return self.encoded_text[index]
    
    def __len__(self):
        return len(self.data)
    

In [ ]:
import tiktoken 
tokenizer = tiktoken.get_encoding("gpt2")

In [ ]:
def custom_collate_fn(
        batch,
        pad_token_id = 50256,
        ignore_index = -100,
        device = 'cpu'
):
    batch_max_len = max(len(item)+1 for item in batch)## added by 1 for the eos token

    input_lst, target_lst = [], []

    for item in batch:
        new_item = item.copy()
        new_item.append(pad_token_id)  ## adding eos token at the end
        padded = (
            new_item + [pad_token_id] * (batch_max_len - len(new_item)) # padding to max length
        )

        inputs = torch.tensor(padded[:-1]) ## removing the extra padded token 
        targets = torch.tensor(padded[1:]) ## shifting by 1 for target

        # replace all but the first padding tokens in targets by ignore_index
        mask= targets == pad_token_id ## create a mask for padding tokens
        indices = torch.nonzero(mask).squeeze() ## get the indices of padding tokens
        if indices.numel()> 1:
            targets[indices[1:]] = ignore_index ## replace all but the first padding token with ignore_index
        
        input_lst.append(inputs)
        target_lst.append(targets)
    ## convert list of inputs to tensor and transfer to target device 
    input_tensor = torch.stack(input_lst).to(device)
    target_tensor = torch.stack(target_lst).to(device)
    return input_tensor, target_tensor

    

In [ ]:
## example usage of the dataset and collate function
input_1 = [0, 1, 2, 3]
input_2 = [4, 5, 6]
input_3 = [7, 8, 9, 10, 11]
batch = [input_1, input_2, input_3]
padded_batch = custom_collate_fn(batch, pad_token_id=50256, device='cpu')
print(padded_batch)

## Creating dataloader 

We moved the data onto the target device (for example, the GPU memory when device="cuda") in the main training loop. Having this as part of the collate function offers the advantage of performing this device transfer process as a background process outside the training loop, preventing it from blocking the GPU during model training.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

Next, to reuse the chosen device setting in custom_collate_fn when we plug it into the PyTorch DataLoader class later in this section, we use the partial function from Python's functools standard library to create a new version of the function with the device argument pre-filled.

Additionally, we set the allowed_max_length to 1024, which truncates the data to the maximum context length supported by the GPT-2 model we finetune later in this chapter:

In [ ]:
from functools import partial
customized_collate_fn = partial(custom_collate_fn, device=device, allowed_max_length=1024)

In [ ]:
from torch.utils.data import Dataloader

num_workers = 0
batch_size = 8

torch.manual_seed(42)

train_dataset = InstructionDataset(
    train_data, tokenizer
)

train_loader = Dataloader(
    train_dataset,
    batch_size = batch_size,
    collate_fn = customized_collate_fn,
    shuffle = True,
    drop_last = True,
    num_workers = num_workers
)

val_data = InstructionDataset(
    val_data, tokenizer
)
val_loader = Dataloader(
    val_data,
    batch_size = batch_size,
    collate_fn = customized_collate_fn,
    shuffle = False,
    drop_last = False,
    num_workers = num_workers
)

test_data = InstructionDataset(
    test_data, tokenizer
)
test_loader = Dataloader(
    test_data,
    batch_size = batch_size,
    collate_fn = customized_collate_fn,
    shuffle = False,
    drop_last = False,
    num_workers = num_workers
)



In [ ]:
print("Train_loder")
for inputs, targets, in train_loader:
    print("Inputs:", inputs)
    print("Targets:", targets)